# Smart Waste Classification — ML Pipeline

## Objective

This notebook develops and evaluates machine-learning models for classifying waste images into five categories:

- Glass
- Metal
- Organic
- Paper
- Plastic

The cleaned dataset contains 1,640 usable images after removing exact duplicate samples and contradictory cross-class duplicates.

Two modelling approaches will be investigated:

1. Classical machine learning using HOG image features.
2. A convolutional neural network using TensorFlow.

The models will be evaluated using accuracy, precision, recall, F1-score, and confusion matrices.

In [2]:
%pip install opencv-python
import cv2 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

import keras
from keras import layers

Note: you may need to restart the kernel to use updated packages.


In [3]:
DATA_DIR = Path("../data/raw")

print("Dataset exists:", DATA_DIR.exists())

Dataset exists: True


In [5]:
df = pd.read_csv("../data/clean_dataset.csv")

print("Total images:", len(df))

Total images: 1640


In [6]:
print(df["class"].value_counts())

class
Metal      373
Plastic    335
Paper      320
Glass      312
Organic    300
Name: count, dtype: int64


In [8]:
X = df["filename"].values
y = df["class"].values

In [9]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['Glass' 'Metal' 'Organic' 'Paper' 'Plastic']


## Train, Validation, and Test Split

The cleaned dataset is divided into training, validation, and test sets using stratified sampling.

The split consists of approximately:

- 70% training data
- 15% validation data
- 15% test data

Stratification ensures that the five waste categories remain proportionally represented across each partition.

The test set is kept separate until final model evaluation.

In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=42
)

In [18]:
X_train.shape

(1148,)

In [14]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [15]:
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 1148
Validation: 246
Testing: 246


In [19]:
split_distribution = pd.DataFrame({
    "Train": pd.Series(y_train).value_counts(normalize=True),
    "Validation": pd.Series(y_val).value_counts(normalize=True),
    "Test": pd.Series(y_test).value_counts(normalize=True)
})

split_distribution

,Train,Validation,Test
1,0.227352,0.227642,0.227642
4,0.204704,0.203252,0.203252
3,0.195122,0.195122,0.195122
0,0.189895,0.191057,0.191057
2,0.182927,0.182927,0.182927
